# Update Data Dictionary

Update the shared dictionary using the latest derived-field registry and bin inventory.

Source-field entries are preserved. Existing derived-field entries are updated, and new fields are added.

The dictionary is saved only after validation passes.

In [1]:
from pathlib import Path
import json

import pandas as pd

def find_repo_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / "data" / "allstate_claims_data.csv").exists():
            return path

    raise FileNotFoundError("Repository root not found.")


ROOT = find_repo_root()

GATE2 = (
    ROOT / "notebooks" / "final-deliverables"
    / "September" / "Gate 2"
)

EVIDENCE = GATE2 / "continuous_analysis_evidence"

DICTIONARY_PATH = GATE2 / "data_dictionary.csv"
REGISTRY_PATH = EVIDENCE / "derived_field_registry.csv"
BIN_PATH = EVIDENCE / "bin_inventory.csv"

NEW_VERSION = "1.1.0"

## Load Current Dictionary and EDA Evidence

In [2]:
dictionary = pd.read_csv(
    DICTIONARY_PATH,
    keep_default_na=False
)

registry = pd.read_csv(
    REGISTRY_PATH,
    keep_default_na=False
)

bins = pd.read_csv(
    BIN_PATH,
    keep_default_na=False
)

print(f"Current dictionary: {len(dictionary)} fields")
print(f"Registered derived fields: {len(registry)}")
print(f"Continuous bin definitions: {len(bins)}")

Current dictionary: 133 fields
Registered derived fields: 15
Continuous bin definitions: 14


## Update Derived Fields

Use the registered definitions and actual bin boundaries. Preserve all source-field entries.

In [3]:
source = dictionary[
    dictionary["field_origin"] == "source"
].copy()

existing_derived = dictionary[
    dictionary["field_origin"] == "derived"
].copy()

# Preserve existing derived-field metadata.
existing_lookup = existing_derived.set_index("field_name")

bin_lookup = bins.set_index("derived_field")

updated_rows = []

for _, field in registry.iterrows():

    name = field["field_name"]

    if name in existing_lookup.index:
        row = existing_lookup.loc[name].to_dict()
    else:
        row = {col: "" for col in dictionary.columns}

    row["field_name"] = name
    row["field_origin"] = "derived"
    row["source_name"] = ""

    # Use the definitions recorded by the EDA workflow.
    for column in [
        "purpose",
        "inputs",
        "exact_derivation",
        "allowed_values",
        "field_version",
        "owner",
    ]:
        row[column] = field[column]

    row["observed_dtype"] = field["observed_dtype"]
    row["n_unique"] = field["observed_unique_count"]
    row["missing_count"] = field["observed_missing_count"]

    row["missing_pct"] = (
        int(field["observed_missing_count"])
        / 188_318 * 100
    )

    if name == "log1p_loss":

        row["approved_role"] = "derived target"
        row["unit_or_scale"] = "log1p transformation of loss"

        row["documented_meaning"] = (
            "Log-transformed final paid claim amount"
        )

        row["quality_limitation"] = (
            "Transformed values are not original claim amounts in dollars."
        )

        row["september_handling"] = (
            "Use for exploratory analysis only. "
            "Retain original loss for reporting and MAE evaluation."
        )

        row["meaning_status"] = "documented"

        # Existing observed statistics are retained.

    elif name in bin_lookup.index:

        info = bin_lookup.loc[name]

        actual_bins = int(info["actual_bins"])
        edges = json.loads(info["bin_edges_json"])

        row["approved_role"] = "derived bin label"

        row["unit_or_scale"] = (
            f"Quantile-bin label; {actual_bins} observed bins"
        )

        row["documented_meaning"] = (
            f"Quantile-bin assignment for {info['field']}"
        )

        row["quality_limitation"] = (
            "Repeated predictor values may prevent 10 distinct bins. "
            "Boundaries depend on the observed source distribution."
        )

        row["september_handling"] = (
            "Use for continuous predictor support and target-pattern "
            "analysis. Do not interpret bin relationships as causal."
        )

        row["meaning_status"] = "documented"

        row["observed_min"] = 1
        row["observed_max"] = actual_bins

        row["category_levels"] = json.dumps({
            "labels": list(range(1, actual_bins + 1)),
            "edges": edges,
        })

    else:
        raise ValueError(
            f"No handling rules defined for derived field: {name}"
        )

    row["dictionary_version"] = NEW_VERSION

    updated_rows.append(row)

updated_derived = pd.DataFrame(
    updated_rows,
    columns=dictionary.columns
)

print(f"Updated derived fields: {len(updated_derived)}")

Updated derived fields: 15


## Combine and Validate

Keep all 132 source fields and include each registered derived field once.

In [4]:
# Verify source-field inventory.
assert len(source) == 132
assert source["field_name"].is_unique

# Verify the registry and bin inventory.
assert registry["field_name"].is_unique
assert bins["derived_field"].is_unique

assert set(bins["derived_field"]).issubset(
    set(registry["field_name"])
)

# Confirm required derived metadata is present.
required = [
    "field_name",
    "purpose",
    "inputs",
    "exact_derivation",
    "allowed_values",
    "field_version",
    "owner",
]

assert registry[required].ne("").all().all()

# Preserve any previously documented derived fields that are not
# included in the current registry.
remaining_derived = existing_derived[
    ~existing_derived["field_name"].isin(registry["field_name"])
]

updated_dictionary = pd.concat(
    [
        source,
        remaining_derived,
        updated_derived,
    ],
    ignore_index=True
)

updated_dictionary["dictionary_version"] = NEW_VERSION

assert updated_dictionary["field_name"].is_unique

assert len(updated_dictionary) == (
    132 + len(remaining_derived) + len(updated_derived)
)

print(f"Source fields: {len(source)}")
print(f"Previously documented derived fields: {len(remaining_derived)}")
print(f"Registered derived fields: {len(updated_derived)}")
print(f"Total fields: {len(updated_dictionary)}")

print("\nDictionary validation: PASS")

Source fields: 132
Previously documented derived fields: 0
Registered derived fields: 15
Total fields: 147

Dictionary validation: PASS


## Review Changes

Check which fields were added or updated before publishing.

In [5]:
new_fields = updated_dictionary[
    ~updated_dictionary["field_name"].isin(
        dictionary["field_name"]
    )
]

print(f"New fields: {len(new_fields)}")

display(
    new_fields[
        [
            "field_name",
            "approved_role",
            "inputs",
            "exact_derivation",
            "allowed_values",
            "owner",
        ]
    ]
)

New fields: 14


,field_name,approved_role,inputs,exact_derivation,allowed_values,owner
133,cont1_quantile_bin,derived bin label,cont1,"pd.qcut(cont1, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
134,cont2_quantile_bin,derived bin label,cont2,"pd.qcut(cont2, q=10, labels=False, duplicates=...",integer bin labels 1..9,Liam
135,cont3_quantile_bin,derived bin label,cont3,"pd.qcut(cont3, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
136,cont4_quantile_bin,derived bin label,cont4,"pd.qcut(cont4, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
137,cont5_quantile_bin,derived bin label,cont5,"pd.qcut(cont5, q=10, labels=False, duplicates=...",integer bin labels 1..8,Liam
138,cont6_quantile_bin,derived bin label,cont6,"pd.qcut(cont6, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
139,cont7_quantile_bin,derived bin label,cont7,"pd.qcut(cont7, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
140,cont8_quantile_bin,derived bin label,cont8,"pd.qcut(cont8, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
141,cont9_quantile_bin,derived bin label,cont9,"pd.qcut(cont9, q=10, labels=False, duplicates=...",integer bin labels 1..10,Liam
142,cont10_quantile_bin,derived bin label,cont10,"pd.qcut(cont10, q=10, labels=False, duplicates...",integer bin labels 1..10,Liam


## Save Updated Dictionary

Publish the new version after reviewing the changes.

In [6]:
updated_dictionary.to_csv(
    DICTIONARY_PATH,
    index=False
)

print(f"Saved dictionary version {NEW_VERSION}")
print(f"Total fields: {len(updated_dictionary)}")
print(DICTIONARY_PATH.relative_to(ROOT))

Saved dictionary version 1.1.0
Total fields: 147
notebooks\final-deliverables\September\Gate 2\data_dictionary.csv
